In [ ]:
import numpy as np
import cv2
import os
from glob import glob
from skimage import io, color, feature
from skimage.feature import greycomatrix, greycoprops
from scipy.ndimage import generic_filter, sobel
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
def load_dataset(image_dir, mask_dir):
    """加载图像和标注mask"""
    images = []
    masks = []
    image_to_mask_mapping = {
    "1.jpg": "C0001273_EX.png",
    "2.jpg": "C0002369_EX.png",
    "3.jpg": "C0003164_EX.png",
    "4.jpg": "C0003166_EX.png",
    "5.jpg": "C0007155_EX.png",
    "6.jpg": "C0007158_EX.png",
    "7.jpg": "C0007248_EX.png",
    "8.jpg": "C0010059_EX.png",
    "9.jpg": "C0010362_EX.png",
    "10.jpg":"C0010452_EX.png",
    "11.jpg":"C0010492_EX.png",
    "12.jpg":"C0014759_EX.png",
    "13.jpg":"C0016193_EX.png",
    "14.jpg":"C0024407_EX.png",
    "15.jpg":"C0026111_EX.png"
}
    # 遍历图像目录中的所有文件
    for img_path in glob(os.path.join(image_dir, "*.*")):
        # 提取图像文件名（去掉路径和扩展名）
        base_name = os.path.basename(img_path)
        
        # 构造对应的标注图文件名
        # 去光照图像的文件名为 "remove_illumination_XXXX.bmp"
        # 对应的标注图为 "XXXX_EX.png"
        if base_name.startswith("remove_illumination_"):
            # 去除前缀 "remove_illumination_" 并替换扩展名为 ".png"
            mask_base_name = base_name.replace("remove_illumination_", "").replace(".bmp", "_EX.png")
        else:
            mask_base_name = image_to_mask_mapping.get(base_name, None)
        mask_path = os.path.join(mask_dir, mask_base_name)
        # 加载图像和标注图
        img = io.imread(img_path)
        mask = io.imread(mask_path)
        # 检查 mask 是否包含两类
        unique_vals = np.unique(mask)
        if len(unique_vals) < 2:
            print(f"Warning: Mask {mask_path} has only one class: {unique_vals}. Skipping.")
            continue
        # 检查文件是否成功加载
        if img is None or mask is None:
            print(f"Warning: Failed to load {img_path} or {mask_path}")
            continue
        
        images.append(img)
        masks.append(mask)
    
    return images, masks

In [ ]:
#  加载训练集和测试集
train_image_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/去光照/训练"
train_mask_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/groudtruth/训练"
test_image_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/去光照/测试"
test_mask_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/groudtruth/测试"

images_train, masks_train = load_dataset(train_image_dir, train_mask_dir)
images_test, masks_test = load_dataset(test_image_dir, test_mask_dir)

In [ ]:
from skimage.transform import resize
from skimage.feature import greycomatrix, greycoprops  # 替换为新函数
def extract_features(img, mask=None):
    """提取优化后的颜色、纹理、空间特征"""
    features = []

    # 缩小图像分辨率（例如 300×200）
    img = resize(img, (300, 200), preserve_range=True).astype(np.uint8)
    if mask is not None:
        mask = resize(mask, (300, 200), preserve_range=True).astype(np.uint8)
        if mask.ndim == 3:
            mask = mask[:, :, 0]  # 如果是 RGB 格式，取第一个通道
        mask = mask.astype(np.uint8)  # 确保是整数类型
        mask[mask > 0] = 1  # 所有非零值设为 1（渗出液）
    # 转换为灰度图
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # 1. 颜色特征（仅保留均值和方差）
    for ch in range(3):
        channel = img[:, :, ch]
        mean_feat = generic_filter(channel, np.mean, size=5, mode="reflect")
        var_feat = generic_filter(channel, np.var, size=5, mode="reflect")
        features.append(mean_feat.flatten())
        features.append(var_feat.flatten())

    # 2. 纹理特征（优化GLCM）
    glcm_props = []
    props_to_use = ["contrast"]  # 只保留一个关键属性
    glcm = greycomatrix(img_gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    
    for prop in props_to_use:
        prop_map = greycoprops(glcm, prop)[0, 0]
        glcm_props.append(np.full_like(img_gray, prop_map, dtype=np.float32).flatten())
    
    features.extend(glcm_props)

    # 3. LBP纹理特征（简化参数）
    lbp = feature.local_binary_pattern(img_gray, P=4, R=1, method="uniform")
    features.append(lbp.flatten())

    # 4. 空间特征（仅保留均值）
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mean_feat = cv2.filter2D(img_gray, -1, kernel/25).flatten()
    features.append(mean_feat)

    X = np.stack(features, axis=1)
    y = mask.flatten() if mask is not None else None
    print(f"Feature shape: {X.shape}")
    return X, y

In [ ]:
def sample_data(X, y, pos_ratio=0.5):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print("Warning: One class has zero samples. Using original data without sampling.")
        return X, y  # 返回原始数据，避免空样本
    
    sample_size = min(len(pos_idx), len(neg_idx))
    pos_sample = np.random.choice(pos_idx, size=int(sample_size*pos_ratio), replace=False)
    neg_sample = np.random.choice(neg_idx, size=int(sample_size*(1-pos_ratio)), replace=False)
    sampled_idx = np.concatenate([pos_sample, neg_sample])
    return X[sampled_idx], y[sampled_idx]

In [ ]:
from sklearn.metrics import accuracy_score, recall_score

# 2. 修改 train_and_evaluate 函数
def train_and_evaluate(images_train, masks_train, images_test, masks_test):
    model = RandomForestClassifier(class_weight="balanced", n_estimators=100, random_state=42)
    
    # 记录训练开始时间
    start_time = time.time()
    
    X_train_all, y_train_all = [], []
    for img, mask in zip(images_train, masks_train):
        X, y = extract_features(img, mask)
        if X is None or len(X) == 0:
            continue
        X_sample, y_sample = sample_data(X, y)
        X_train_all.extend(X_sample)
        y_train_all.extend(y_sample)
    
    if len(X_train_all) == 0:
        raise ValueError("No training samples extracted. Check feature extraction logic.")
    
    X_train_all = np.array(X_train_all)
    y_train_all = np.array(y_train_all)
    
    # 模型训练
    model.fit(X_train_all, y_train_all)
    
    # 计算学习时间
    training_time = time.time() - start_time
    
    # 测试集评估
    X_test_all, y_test_all = [], []
    for img, mask in zip(images_test, masks_test):
        X, y = extract_features(img, mask)
        X_test_all.extend(X)
        y_test_all.extend(y)
    
    X_test_all = np.array(X_test_all)
    y_test_all = np.array(y_test_all)
    
    # 预测
    y_pred = model.predict(X_test_all)
    y_score = model.predict_proba(X_test_all)[:, 1]
    
    # 计算评估指标
    accuracy = accuracy_score(y_test_all, y_pred)
    recall = recall_score(y_test_all, y_pred, pos_label=1, average='binary')
    cm = confusion_matrix(y_test_all, y_pred)
    dice = f1_score(y_test_all, y_pred)
    # ROC曲线数据
    fpr, tpr, _ = roc_curve(y_test_all, y_score)
    roc_auc = roc_auc_score(y_test_all, y_score)
    
    return model, cm, accuracy, recall, (fpr, tpr, roc_auc), training_time,dice

In [ ]:
print("训练图像数量:", len(images_train))
print("测试图像数量:", len(images_test))
print("训练mask数量:", len(masks_train))
print("测试mask数量:", len(masks_test))
import warnings
warnings.filterwarnings("ignore", message="Precision loss occurred in moment calculation")

In [ ]:
# 3. 修改模型评估输出部分
import time
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# 训练和评估模型
model, cm, accuracy, recall, (fpr, tpr, roc_auc), training_time,dice = train_and_evaluate(
    images_train, masks_train, images_test, masks_test
)

# 输出评估结果
print("Confusion Matrix:")
print(cm)
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Training Time: {training_time:.2f} seconds")

# 绘制ROC曲线
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(ticks=[0.5, 1.5], labels=["Background", "Exudate"])
plt.yticks(ticks=[0.5, 1.5], labels=["Background", "Exudate"])
plt.show()

In [ ]:
import joblib

# 保存模型到本地
model=joblib.load( "exudate_segmentation_model.pkl")
print("Model loaded as 'exudate_segmentation_model.pkl'")

In [ ]:
def plot_rf_predictions_white(model, images_test, masks_test, num_samples=3):
    """
    随机森林预测结果可视化（白色显示）
    :param model: 训练好的随机森林分类器
    :param images_test: 测试图像列表 (原始尺寸)
    :param masks_test: 测试标注列表 (原始尺寸)
    :param num_samples: 显示样本数
    """
    plt.figure(figsize=(15, 5*num_samples))
    
    for idx in range(num_samples):
        if idx >= len(images_test):
            break
            
        # 获取测试样本
        img_original = images_test[idx]
        mask_original = masks_test[idx]
        
        # 预处理（需要与训练时一致）
        img_resized = resize(img_original, (300, 200), preserve_range=True).astype(np.uint8)
        mask_resized = resize(mask_original, (300, 200), preserve_range=True).astype(np.uint8)
        mask_resized = (mask_resized > 0).astype(np.uint8)  # 二值化
        
        # 特征提取与预测
        X, _ = extract_features(img_original, mask=None)
        y_pred = model.predict(X)
        pred_mask = y_pred.reshape(300, 200).astype(np.uint8)
        # 创建子图画布
        ax1 = plt.subplot(num_samples, 3, idx*3+1)
        ax2 = plt.subplot(num_samples, 3, idx*3+2)
        ax3 = plt.subplot(num_samples, 3, idx*3+3)
        
        # 原始图像
        ax1.imshow(img_resized, cmap='gray' if img_resized.ndim==2 else None)
        ax1.set_title(f"Sample {idx+1}\nOriginal Image")
        ax1.axis('off')
        
        # 真实标注 (白色显示)
        ax2.imshow(mask_resized, cmap='gray')
        ax2.set_title("Ground Truth")
        ax2.axis('off')
        
        # 预测结果 (白色显示)
        ax3.imshow(pred_mask, cmap='gray')
        ax3.set_title("Prediction")
        ax3.axis('off')
    
    plt.tight_layout()
    plt.show()

# 使用示例
plot_rf_predictions_white(model, images_test, masks_test, num_samples=3)

In [ ]:
import pyvista as pv
import numpy as np
from matplotlib.cm import get_cmap

def plot_3d_segmentation(volume_3d, spacing=(1,1,3), opacity=0.3, color='red'):
    """修正后的三维可视化函数"""
    # 创建网格对象 (ImageData)
    grid = pv.ImageData()
    
    # 设置网格参数（关键修正点）
    grid.dimensions = np.array(volume_3d.shape)  # 维度应为数据形状（非形状+1）
    grid.spacing = spacing  # 物理间距
    grid.origin = (0, 0, 0)  
    
    # 将数据关联到顶点（Point Data）
    grid.point_data["labels"] = volume_3d.flatten(order="F")  # 注意Fortran顺序
    
    # 提取等值面
    contours = grid.contour([0.5], scalars="labels")  # 显式指定标量场
    
    # 可视化设置
    p = pv.Plotter()
    p.add_mesh(contours, 
               color=color, 
               opacity=opacity,
               smooth_shading=True)
    p.show()

In [ ]:
X_test_all, y_test_all = [], []
for img, mask in zip(images_test, masks_test):
    X, y = extract_features(img, mask)
    X_test_all.extend(X)
    y_test_all.extend(y)
    
X_test_all = np.array(X_test_all)
y_test_all = np.array(y_test_all)
    
    # 预测
y_pred = model.predict(X_test_all)

In [ ]:
rf_pred_3d = y_pred.reshape(300, 200, 15)

# 生成三维渲染（假设物理间距0.3mm×0.3mm×2mm）
plot_3d_segmentation(rf_pred_3d, 
                    spacing=(0.3, 0.3, 2.0),  
                    opacity=0.6,
                    color='#FF4500')

In [ ]:
import pyvista as pv
print(pv.__version__)  # 需 ≥ 0.32.0